In [4]:
import sys

print("Python version:")
print(sys.version)

print("\nPython environment:")
print(sys.executable)

Python version:
3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]

Python environment:
/usr/bin/python3


In [5]:
!pip install -q pandas numpy scikit-learn joblib nltk

In [6]:
import pandas as pd
import numpy as np
import re
import joblib
import nltk

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

print("All libraries imported successfully.")

All libraries imported successfully.


In [8]:
!wget -q https://onlineacademiccommunity.uvic.ca/isot/wp-content/uploads/sites/7295/2023/02/ISOT_Fake_News_Dataset.zip

In [9]:
!unzip -o ISOT_Fake_News_Dataset.zip

unzip:  cannot find or open ISOT_Fake_News_Dataset.zip, ISOT_Fake_News_Dataset.zip.zip or ISOT_Fake_News_Dataset.zip.ZIP.


In [15]:
!pip install -q kagglehub

In [16]:
import kagglehub

dataset_path = kagglehub.dataset_download(
    "clmentbisaillon/fake-and-real-news-dataset"
)

print("Dataset downloaded to:")
print(dataset_path)

Using Colab cache for faster access to the 'fake-and-real-news-dataset' dataset.
Dataset downloaded to:
/kaggle/input/fake-and-real-news-dataset


In [17]:
import os

for root, folders, files in os.walk(dataset_path):

    for file in files:

        print(os.path.join(root, file))

/kaggle/input/fake-and-real-news-dataset/True.csv
/kaggle/input/fake-and-real-news-dataset/Fake.csv


In [18]:
import pandas as pd

fake_df = pd.read_csv(
    "/kaggle/input/fake-and-real-news-dataset/Fake.csv"
)

real_df = pd.read_csv(
    "/kaggle/input/fake-and-real-news-dataset/True.csv"
)

print("Fake news:", fake_df.shape)
print("Real news:", real_df.shape)

Fake news: (23481, 4)
Real news: (21417, 4)


In [19]:
print("Fake dataset columns:")
print(fake_df.columns.tolist())

print("\nReal dataset columns:")
print(real_df.columns.tolist())

print("\nFake news sample:")
display(fake_df.head(3))

print("\nReal news sample:")
display(real_df.head(3))

Fake dataset columns:
['title', 'text', 'subject', 'date']

Real dataset columns:
['title', 'text', 'subject', 'date']

Fake news sample:


,title,text,subject,date
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017"
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017"
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017"



Real news sample:


,title,text,subject,date
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017"
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017"
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017"


In [20]:
fake_df["label"] = 0
real_df["label"] = 1

print("Fake label:", fake_df["label"].unique())
print("Real label:", real_df["label"].unique())

Fake label: [0]
Real label: [1]


In [21]:
df = pd.concat(
    [fake_df, real_df],
    ignore_index=True
)

print("Combined dataset shape:")
print(df.shape)

print("\nLabel distribution:")
print(df["label"].value_counts())

Combined dataset shape:
(44898, 5)

Label distribution:
label
0    23481
1    21417
Name: count, dtype: int64


In [22]:
df["text"] = (
    df["title"].fillna("")
    + " "
    + df["text"].fillna("")
)

df = df[
    ["text", "label"]
]

print(df.head())
print("\nShape:", df.shape)

                                                text  label
0   Donald Trump Sends Out Embarrassing New Year’...      0
1   Drunk Bragging Trump Staffer Started Russian ...      0
2   Sheriff David Clarke Becomes An Internet Joke...      0
3   Trump Is So Obsessed He Even Has Obama’s Name...      0
4   Pope Francis Just Called Out Donald Trump Dur...      0

Shape: (44898, 2)


In [23]:
df["text"] = df["text"].str.strip()

df = df[
    df["text"] != ""
]

df = df.reset_index(drop=True)

print("Final rows:", len(df))

Final rows: 44898


In [24]:
df = df.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

display(df.head())

,text,label
0,Ben Stein Calls Out 9th Circuit Court: Committ...,0
1,Trump drops Steve Bannon from National Securit...,1
2,Puerto Rico expects U.S. to lift Jones Act shi...,1
3,OOPS: Trump Just Accidentally Confirmed He Lea...,0
4,Donald Trump heads for Scotland to reopen a go...,1


In [25]:
df.to_csv(
    "fake_news.csv",
    index=False
)

print("Permanent dataset created.")
print("Rows:", len(df))
print("Columns:", df.columns.tolist())

Permanent dataset created.
Rows: 44898
Columns: ['text', 'label']


In [26]:
import re

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC

In [27]:
def clean_text(text):

    text = str(text)

    # Convert to lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(
        r"https?://\S+|www\.\S+",
        " ",
        text
    )

    # Remove extra whitespace
    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip()


df["text"] = df["text"].apply(clean_text)

print(df["text"].head())

0    ben stein calls out 9th circuit court: committ...
1    trump drops steve bannon from national securit...
2    puerto rico expects u.s. to lift jones act shi...
3    oops: trump just accidentally confirmed he lea...
4    donald trump heads for scotland to reopen a go...
Name: text, dtype: object


In [28]:
X = df["text"]
y = df["label"]

print("Number of articles:", len(X))
print("Number of labels:", len(y))

Number of articles: 44898
Number of labels: 44898


In [29]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training articles:", len(X_train))
print("Testing articles:", len(X_test))

Training articles: 35918
Testing articles: 8980


In [30]:
vectorizer = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1, 2),
    sublinear_tf=True,
    min_df=2,
    max_df=0.95
)

print("TF-IDF vectorizer created.")

TF-IDF vectorizer created.


In [31]:
X_train_tfidf = vectorizer.fit_transform(
    X_train
)

print(
    "Training vector shape:",
    X_train_tfidf.shape
)

Training vector shape: (35918, 50000)


In [32]:
X_test_tfidf = vectorizer.transform(
    X_test
)

print(
    "Testing vector shape:",
    X_test_tfidf.shape
)

Testing vector shape: (8980, 50000)


In [33]:
model = LinearSVC(
    C=1.0,
    random_state=42
)

print("Training SVM...")

Training SVM...


In [34]:
model.fit(
    X_train_tfidf,
    y_train
)

print("Training completed.")

Training completed.


In [35]:
y_pred = model.predict(
    X_test_tfidf
)

print("Predictions completed.")

Predictions completed.


In [36]:
accuracy = accuracy_score(
    y_test,
    y_pred
)

print(
    f"Accuracy: {accuracy * 100:.2f}%"
)

Accuracy: 99.76%


In [37]:
precision = precision_score(
    y_test,
    y_pred
)

recall = recall_score(
    y_test,
    y_pred
)

f1 = f1_score(
    y_test,
    y_pred
)

print(f"Accuracy : {accuracy * 100:.2f}%")
print(f"Precision: {precision * 100:.2f}%")
print(f"Recall   : {recall * 100:.2f}%")
print(f"F1 Score : {f1 * 100:.2f}%")

Accuracy : 99.76%
Precision: 99.72%
Recall   : 99.77%
F1 Score : 99.74%


In [38]:
print(
    classification_report(
        y_test,
        y_pred,
        target_names=[
            "Fake",
            "Real"
        ]
    )
)

              precision    recall  f1-score   support

        Fake       1.00      1.00      1.00      4696
        Real       1.00      1.00      1.00      4284

    accuracy                           1.00      8980
   macro avg       1.00      1.00      1.00      8980
weighted avg       1.00      1.00      1.00      8980



In [39]:
cm = confusion_matrix(
    y_test,
    y_pred
)

print("Confusion Matrix:")
print(cm)

Confusion Matrix:
[[4684   12]
 [  10 4274]]


In [40]:
test_news = """
The government announced a new national education policy
that will provide additional financial support to students.
"""

test_news = clean_text(
    test_news
)

test_vector = vectorizer.transform(
    [test_news]
)

prediction = model.predict(
    test_vector
)[0]

if prediction == 0:
    print("Prediction: FAKE")
else:
    print("Prediction: REAL")

Prediction: FAKE


In [41]:
import os

os.makedirs(
    "verifi_models",
    exist_ok=True
)

joblib.dump(
    model,
    "verifi_models/fake_news_model.pkl"
)

joblib.dump(
    vectorizer,
    "verifi_models/tfidf_vectorizer.pkl"
)

print("Model saved successfully.")

Model saved successfully.


In [42]:
import os

print(
    os.listdir("verifi_models")
)

['tfidf_vectorizer.pkl', 'fake_news_model.pkl']


In [45]:
import zipfile
import os

files_to_save = [
    "fake_news.csv",
    "verifi_models/fake_news_model.pkl",
    "verifi_models/tfidf_vectorizer.pkl"
]

with zipfile.ZipFile(
    "verifi_artifacts.zip",
    "w",
    zipfile.ZIP_DEFLATED
) as zip_file:

    for file in files_to_save:
        zip_file.write(
            file,
            arcname=os.path.basename(file)
        )

print("Created:", os.path.abspath("verifi_artifacts.zip"))

Created: /content/verifi_artifacts.zip


In [46]:
import os

print(os.path.exists("verifi_artifacts.zip"))
print(os.path.getsize("verifi_artifacts.zip"))

True
45519606


In [47]:
import os

for file in os.listdir("/content"):
    print(file)

.config
verifi_artifacts.zip
fake_news.csv
verifi_models
sample_data


In [48]:
from pathlib import Path

zip_path = Path("/content/verifi_artifacts.zip")

print(zip_path)
print(zip_path.exists())
print(zip_path.stat().st_size)

/content/verifi_artifacts.zip
True
45519606


In [49]:
import shutil
import os

shutil.copy(
    "/content/fake_news.csv",
    "/content/verifi_dataset.csv"
)

shutil.copy(
    "/content/verifi_models/fake_news_model.pkl",
    "/content/verifi_fake_news_model.pkl"
)

shutil.copy(
    "/content/verifi_models/tfidf_vectorizer.pkl",
    "/content/verifi_tfidf_vectorizer.pkl"
)

print("Files prepared:")
print(os.listdir("/content"))

Files prepared:
['.config', 'verifi_artifacts.zip', 'verifi_dataset.csv', 'verifi_fake_news_model.pkl', 'fake_news.csv', 'verifi_models', 'verifi_tfidf_vectorizer.pkl', 'sample_data']


In [54]:
import os

for file in os.listdir("/content"):
    if file.endswith(".pkl"):
        print(file)

tfidf_vectorizer.pkl
fake_news_model.pkl
verifi_fake_news_model.pkl
verifi_tfidf_vectorizer.pkl


In [55]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [56]:
import shutil
import os

drive_folder = "/content/drive/MyDrive/Verifi_Models"

os.makedirs(
    drive_folder,
    exist_ok=True
)

shutil.copy(
    "/content/verifi_models/fake_news_model.pkl",
    drive_folder
)

shutil.copy(
    "/content/verifi_models/tfidf_vectorizer.pkl",
    drive_folder
)

print("Files copied to Google Drive.")
print(os.listdir(drive_folder))

Files copied to Google Drive.
['fake_news_model.pkl', 'tfidf_vectorizer.pkl']
